# HookTheory preprocessing pipeline

Ноутбук запускает полную цепочку подготовки данных в `src/data`:

1. `preprocess_hooktheory.py` — из raw JSON делает очищенный `hooktheory_processed.json`, `hooktheory_processed_structured_only.json` и служебные файлы unmatched/unknown split.
2. `build_preprocess_song_timelines.py` — группирует structured-only клипы по `ori_uid` и строит `original_songs_timeline.json`.
3. `canonicalize_hooktheory.py` — нормализует symbolic поля и сохраняет `hooktheory_canonical.json` для полного и structured-only датасета.
4. `encode_teacher_features.py` — переводит canonical JSON в финальный численный формат `teacher_encoded.json`, готовый для dataloader и graph builder.

Пайплайн имеет такой вид:

raw HookTheory
-> hooktheory_processed.json
-> original_songs_timeline.json
-> hooktheory_canonical.json
-> teacher_encoded.json

In [1]:
from pathlib import Path
import subprocess
import shlex

# === Настройки путей ===
REPO_ROOT = Path.cwd()
SCRIPTS_DIR = REPO_ROOT / 'src' / 'data'
RAW_DATA_DIR = REPO_ROOT / 'data' / 'HookTheory'
CANONICAL_DATA_DIR = REPO_ROOT / 'data' / 'HTCanon'
METADATA_DIR = REPO_ROOT / 'metadata'

RAW_JSON = Path(RAW_DATA_DIR, 'Hooktheory_Raw.json', '4_merged.json')
STRUCTURE_TRAIN = Path(RAW_DATA_DIR, 'HookTheoryStructure.train.jsonl')
STRUCTURE_VAL = Path(RAW_DATA_DIR, 'HookTheoryStructure.val.jsonl')
STRUCTURE_TEST = Path(RAW_DATA_DIR, 'HookTheoryStructure.test.jsonl')

OUT_DIR = Path(CANONICAL_DATA_DIR, 'HK_processed')
CANONICAL_FULL_DIR = OUT_DIR / 'canonical_full'
CANONICAL_STRUCTURED_DIR = OUT_DIR / 'canonical_structured_only'

ENCODED_FULL_DIR = OUT_DIR / 'encoded_full'
ENCODED_STRUCTURED_DIR = OUT_DIR / 'encoded_structured_only'

OUT_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_FULL_DIR.mkdir(parents=True, exist_ok=True)
CANONICAL_STRUCTURED_DIR.mkdir(parents=True, exist_ok=True)
ENCODED_FULL_DIR.mkdir(parents=True, exist_ok=True)
ENCODED_STRUCTURED_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('SCRIPTS_DIR:', SCRIPTS_DIR)
print('METADATA_DIR:', METADATA_DIR)
print('OUT_DIR:', OUT_DIR)

REPO_ROOT: /home/str/HSE/diploma
SCRIPTS_DIR: /home/str/HSE/diploma/src/data
METADATA_DIR: /home/str/HSE/diploma/metadata
OUT_DIR: /home/str/HSE/diploma/data/HTCanon/HK_processed


In [2]:
def run_cmd(cmd: list[str], cwd: Path | None = None):
    print('\n[RUN]', ' '.join(shlex.quote(x) for x in cmd))
    result = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with code {result.returncode}')

def check_files(paths):
    for p in paths:
        print(f"{'OK' if p.exists() else 'MISSING'}: {p}")

In [3]:
# 1) preprocess_hooktheory: raw -> cleaned json
run_cmd([
    'python', str(SCRIPTS_DIR / 'preprocess_hooktheory.py'),
    '--raw-json', str(RAW_JSON),
    '--out-dir', str(OUT_DIR),
    '--structure-train', str(STRUCTURE_TRAIN),
    '--structure-val', str(STRUCTURE_VAL),
    '--structure-test', str(STRUCTURE_TEST),
    '--compute-stats',
])

processed_json = OUT_DIR / 'hooktheory_processed.json'
structured_only_json = OUT_DIR / 'hooktheory_processed_structured_only.json'
unmatched_json = OUT_DIR / 'hooktheory_processed_unmatched_ids.json'
unknown_split_json = OUT_DIR / 'hooktheory_processed_unknown_split_ids.json'
processed_stats_json = OUT_DIR / 'hooktheory_processed.stats.json'

check_files([
    processed_json,
    structured_only_json,
    unmatched_json,
    unknown_split_json,
    processed_stats_json,
])


[RUN] python /home/str/HSE/diploma/src/data/preprocess_hooktheory.py --raw-json /home/str/HSE/diploma/data/HookTheory/Hooktheory_Raw.json/4_merged.json --out-dir /home/str/HSE/diploma/data/HTCanon/HK_processed --structure-train /home/str/HSE/diploma/data/HookTheory/HookTheoryStructure.train.jsonl --structure-val /home/str/HSE/diploma/data/HookTheory/HookTheoryStructure.val.jsonl --structure-test /home/str/HSE/diploma/data/HookTheory/HookTheoryStructure.test.jsonl --compute-stats
[DEBUG] split=train
  processed_ids:  21230
  structure_ids:  9498
  intersection:   9498
  processed_only: 11732
  structure_only: 0
[DEBUG] split=val
  processed_ids:  2184
  structure_ids:  927
  intersection:   927
  processed_only: 1257
  structure_only: 0
[DEBUG] split=test
  processed_ids:  2761
  structure_ids:  1090
  intersection:   1090
  processed_only: 1671
  structure_only: 0
[INFO] saved processed dataset to: /home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed.json
[INFO] saved 

In [4]:
# 2) build_preprocess_song_timelines: объединяем клипы с structure в таймлайны оригинальных песен
run_cmd([
    'python', str(SCRIPTS_DIR / 'build_preprocess_song_timelines.py'),
    '--processed-json', str(structured_only_json),
    '--out-dir', str(OUT_DIR),
    '--compute-stats',
])

timeline_json = OUT_DIR / 'original_songs_timeline.json'
timeline_agg_stats = OUT_DIR / 'original_songs_aggregate.stats.json'
timeline_stats = OUT_DIR / 'original_songs_timeline.stats.json'

check_files([timeline_json, timeline_agg_stats, timeline_stats])


[RUN] python /home/str/HSE/diploma/src/data/build_preprocess_song_timelines.py --processed-json /home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed_structured_only.json --out-dir /home/str/HSE/diploma/data/HTCanon/HK_processed --compute-stats
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_timeline.json
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_aggregate.stats.json
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_timeline.stats.json
[INFO] done

OK: /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_timeline.json
OK: /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_aggregate.stats.json
OK: /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_timeline.stats.json


In [5]:
# 3a) canonicalize полного очищенного JSON
run_cmd([
    'python', str(SCRIPTS_DIR / 'canonicalize_hooktheory.py'),
    '--input', str(processed_json),
    '--out-dir', str(CANONICAL_FULL_DIR),
])

# 3b) canonicalize structured_only очищенного JSON
run_cmd([
    'python', str(SCRIPTS_DIR / 'canonicalize_hooktheory.py'),
    '--input', str(structured_only_json),
    '--out-dir', str(CANONICAL_STRUCTURED_DIR),
])

check_files([
    CANONICAL_FULL_DIR / 'hooktheory_canonical.json',
    CANONICAL_FULL_DIR / 'hooktheory_canonical.stats.json',
    CANONICAL_FULL_DIR / 'hooktheory_canonical.report.json',
    CANONICAL_STRUCTURED_DIR / 'hooktheory_canonical.json',
    CANONICAL_STRUCTURED_DIR / 'hooktheory_canonical.stats.json',
    CANONICAL_STRUCTURED_DIR / 'hooktheory_canonical.report.json',
])


[RUN] python /home/str/HSE/diploma/src/data/canonicalize_hooktheory.py --input /home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed.json --out-dir /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full
[INFO] done
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.json
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.stats.json
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.report.json


[RUN] python /home/str/HSE/diploma/src/data/canonicalize_hooktheory.py --input /home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed_structured_only.json --out-dir /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only
[INFO] done
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only/hooktheory_canonical.json
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_proces

In [6]:
# Просмотр всех созданных JSON-файлов
for p in sorted(OUT_DIR.rglob('*.json')):
    print(p)

/home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.report.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.stats.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only/hooktheory_canonical.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only/hooktheory_canonical.report.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only/hooktheory_canonical.stats.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed.stats.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed_structured_only.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed_unknown_split_ids.json
/home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed_un

In [8]:
# 4a) encode canonical full -> teacher_encoded
run_cmd([
    'python', str(SCRIPTS_DIR / 'encode_teacher_features.py'),
    '--input', str(CANONICAL_FULL_DIR / 'hooktheory_canonical.json'),
    '--metadata-dir', str(METADATA_DIR),
    '--out-dir', str(ENCODED_FULL_DIR),
])

# 4b) encode canonical structured_only -> teacher_encoded
run_cmd([
    'python', str(SCRIPTS_DIR / 'encode_teacher_features.py'),
    '--input', str(CANONICAL_STRUCTURED_DIR / 'hooktheory_canonical.json'),
    '--metadata-dir', str(METADATA_DIR),
    '--out-dir', str(ENCODED_STRUCTURED_DIR),
])

check_files([
    ENCODED_FULL_DIR / 'teacher_encoded.json',
    ENCODED_FULL_DIR / 'teacher_encoded.stats.json',
    ENCODED_FULL_DIR / 'teacher_encoder_maps.json',
    ENCODED_STRUCTURED_DIR / 'teacher_encoded.json',
    ENCODED_STRUCTURED_DIR / 'teacher_encoded.stats.json',
    ENCODED_STRUCTURED_DIR / 'teacher_encoder_maps.json',
])


[RUN] python /home/str/HSE/diploma/src/data/encode_teacher_features.py --input /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.json --metadata-dir /home/str/HSE/diploma/metadata --out-dir /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_full
[INFO] done
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_full/teacher_encoded.json
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_full/teacher_encoded.stats.json
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_full/teacher_encoder_maps.json


[RUN] python /home/str/HSE/diploma/src/data/encode_teacher_features.py --input /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only/hooktheory_canonical.json --metadata-dir /home/str/HSE/diploma/metadata --out-dir /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_structured_only
[INFO] done
[INFO] saved: /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_structured_o

In [9]:
# 5) Очистка служебных stats/report файлов
to_delete = sorted(
    list(OUT_DIR.rglob('*.stats.json')) +
    list(OUT_DIR.rglob('*.report.json'))
)
print(f'Будет удалено файлов: {len(to_delete)}')
for p in to_delete:
    print('DELETE', p)
    p.unlink(missing_ok=True)

print('Готово.')

Будет удалено файлов: 9
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.report.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_full/hooktheory_canonical.stats.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only/hooktheory_canonical.report.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/canonical_structured_only/hooktheory_canonical.stats.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_full/teacher_encoded.stats.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/encoded_structured_only/teacher_encoded.stats.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/hooktheory_processed.stats.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_aggregate.stats.json
DELETE /home/str/HSE/diploma/data/HTCanon/HK_processed/original_songs_timeline.stats.json
Готово.
